In [0]:
# Databricks notebook source
# src/pipelines/silver_pipeline.py

import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =========================================================================
# 1. SILVER CLICKSTREAM (Flattening, Sessionisation, and Quarantine)
# =========================================================================

# Step 1A: Clean & Flatten Clickstream (Temporary intermediate view)
@dlt.table(
    name="silver_clickstream_clean",
    comment="Intermediate cleansed clickstream with extracted metadata",
    temporary=True
)
@dlt.expect_or_drop("valid_user_id", "user_id IS NOT NULL")
@dlt.expect_or_drop("valid_timestamp", "event_timestamp IS NOT NULL")
@dlt.expect("valid_event_type", "event IN ('search', 'view', 'booking_attempt', 'confirm_booking', 'click')")
def silver_clickstream_clean():
    return (
        dlt.read_stream("bronze_clickstream")
           .select(
               F.col("user_id").cast("bigint").alias("user_id"),
               F.col("property_id").cast("bigint").alias("property_id"),
               F.col("event").cast("string").alias("event"),
               F.col("timestamp").cast("timestamp").alias("event_timestamp"),
               # Extract nested metadata safely with fallback
               F.coalesce(F.col("metadata.device"), F.lit("unknown")).alias("device_type"),
               F.coalesce(F.col("metadata.referrer"), F.lit("direct")).alias("referrer"),
               F.col("_source_file"),
               F.col("_batch_id")
           )
    )

# Step 1B: Quarantine for Clickstream -> ops schema
@dlt.table(
    name="ops.quarantine_clickstream",
    comment="Quarantined raw clickstream records failing validity expectations"
)
def quarantine_clickstream():
    return (
        dlt.read_stream("bronze_clickstream")
           .filter("user_id IS NULL OR timestamp IS NULL")
           .withColumn("quarantine_reason", F.lit("Null user_id or event timestamp"))
           .withColumn("quarantined_at", F.current_timestamp())
    )

# Step 1C: 30-Minute Inactivity Sessionisation -> silver schema
@dlt.table(
    name="silver.silver_clickstream",
    comment="Final sessionised clickstream. 30-minute inactivity breaks define distinct sessions."
)
def silver_clickstream():
    clean_df = dlt.read("silver_clickstream_clean")
    
    # 30-minute inactivity window: compute previous event timestamp per user
    user_window = Window.partitionBy("user_id").orderBy("event_timestamp")
    
    with_lag = clean_df.withColumn(
        "prev_event_time", 
        F.lag("event_timestamp").over(user_window)
    )
    
    # 1800 seconds = 30 minutes inactivity gap
    with_gap = with_lag.withColumn(
        "time_diff_sec",
        F.coalesce(F.unix_timestamp("event_timestamp") - F.unix_timestamp("prev_event_time"), F.lit(999999))
    )
    
    # Flag session boundaries
    with_flags = with_gap.withColumn(
        "is_new_session",
        F.when(F.col("time_diff_sec") > 1800, 1).otherwise(0)
    )
    
    # Cumulative sum generates distinct session sequence number per user
    with_session_seq = with_flags.withColumn(
        "session_seq",
        F.sum("is_new_session").over(user_window)
    )
    
    # Unique surrogate key: <user_id>_<session_seq>
    return (
        with_session_seq.withColumn(
            "session_id",
            F.concat(F.col("user_id").cast("string"), F.lit("_s"), F.col("session_seq").cast("string"))
        ).select(
            "session_id",
            "user_id",
            "property_id",
            "event",
            "device_type",
            "referrer",
            "event_timestamp",
            "time_diff_sec",
            "_source_file",
            "_batch_id"
        )
    )


# =========================================================================
# 2. SILVER REVIEWS (Validation, Soft-Delete Tracking, and Quarantine)
# =========================================================================

# Silver reviews -> silver schema
@dlt.table(
    name="silver.silver_reviews",
    comment="Cleansed reviews with preserved soft-deletion flags"
)
@dlt.expect_or_drop("valid_review_id", "review_id IS NOT NULL")
@dlt.expect_or_drop("valid_rating_range", "rating >= 1.0 AND rating <= 5.0")
@dlt.expect_or_fail("valid_status", "is_deleted IS NOT NULL")
def silver_reviews():
    return (
        dlt.read_stream("bronze_reviews")
           .select(
               F.col("review_id").cast("bigint").alias("review_id"),
               F.col("booking_id").cast("bigint").alias("booking_id"),
               F.col("user_id").cast("bigint").alias("user_id"),
               F.col("property_id").cast("bigint").alias("property_id"),
               F.col("rating").cast("double").alias("rating"),
               F.col("comment").cast("string").alias("review_comment"),
               F.col("is_deleted").cast("boolean").alias("is_deleted"),
               F.col("created_at").cast("timestamp").alias("created_at"),
               F.col("updated_at").cast("timestamp").alias("updated_at"),
               F.col("_source_file"),
               F.col("_batch_id")
           )
    )

# Quarantine reviews -> ops schema
@dlt.table(
    name="ops.quarantine_reviews",
    comment="Quarantined reviews with out-of-range ratings or null IDs"
)
def quarantine_reviews():
    return (
        dlt.read_stream("bronze_reviews")
           .filter("review_id IS NULL OR rating < 1.0 OR rating > 5.0")
           .withColumn("quarantine_reason", F.lit("Rating outside 1-5 range or null review_id"))
           .withColumn("quarantined_at", F.current_timestamp())
    )


# =========================================================================
# 3. SILVER BOOKINGS (Deduplication, Validation, and Quarantine)
# =========================================================================

# Silver bookings -> silver schema
@dlt.table(
    name="silver.silver_bookings",
    comment="Cleansed and deduplicated bookings conversion records"
)
@dlt.expect_or_fail("valid_booking_id", "booking_id IS NOT NULL")
@dlt.expect_or_drop("valid_positive_amount", "total_amount > 0")
@dlt.expect_or_drop("valid_date_range", "check_out >= check_in")
def silver_bookings():
    window_dedup = Window.partitionBy("booking_id").orderBy(F.col("updated_at").desc(), F.col("_ingest_time").desc())
    
    return (
        dlt.read("bronze_bookings")
           .withColumn("rn", F.row_number().over(window_dedup))
           .filter(F.col("rn") == 1)
           .drop("rn")
           .select(
               F.col("booking_id").cast("bigint").alias("booking_id"),
               F.col("user_id").cast("bigint").alias("user_id"),
               F.col("property_id").cast("bigint").alias("property_id"),
               F.col("check_in").cast("date").alias("check_in"),
               F.col("check_out").cast("date").alias("check_out"),
               F.col("guests_count").cast("int").alias("guests_count"),
               F.col("total_amount").cast("double").alias("total_amount"),
               F.col("status").cast("string").alias("booking_status"),
               F.col("created_at").cast("timestamp").alias("created_at"),
               F.col("updated_at").cast("timestamp").alias("updated_at"),
               F.col("_source_file"),
               F.col("_batch_id")
           )
    )

# Quarantine bookings -> ops schema
@dlt.table(
    name="ops.quarantine_bookings",
    comment="Quarantined bookings with negative amounts or inverted dates"
)
def quarantine_bookings():
    return (
        dlt.read_stream("bronze_bookings")
           .filter("total_amount <= 0 OR check_out < check_in")
           .withColumn("quarantine_reason", F.lit("Negative total_amount or checkout before checkin"))
           .withColumn("quarantined_at", F.current_timestamp())
    )


# =========================================================================
# 4. DIM PROPERTIES -> silver schema
# =========================================================================

@dlt.table(
    name="silver.dim_properties",
    comment="Conformed properties dimension deduplicated on property_id"
)
def dim_properties():
    window_prop = Window.partitionBy("property_id").orderBy(F.col("_ingest_time").desc())
    return (
        dlt.read("bronze_properties")
           .withColumn("rn", F.row_number().over(window_prop))
           .filter(F.col("rn") == 1)
           .drop("rn")
           .select(
               F.col("property_id").cast("bigint").alias("property_id"),
               F.col("host_id").cast("bigint").alias("host_id"),
               F.col("destination_id").cast("bigint").alias("destination_id"),
               F.col("title").cast("string").alias("title"),
               F.col("description").cast("string").alias("description"),
               F.col("base_price").cast("double").alias("base_price"),
               F.col("property_type").cast("string").alias("property_type"),
               F.col("max_guests").cast("int").alias("max_guests"),
               F.col("bedrooms").cast("int").alias("bedrooms"),
               F.col("bathrooms").cast("int").alias("bathrooms"),
               F.col("property_latitude").cast("double").alias("property_latitude"),
               F.col("property_longitude").cast("double").alias("property_longitude"),
               F.col("created_at").cast("date").alias("created_at")
           )
    )


# =========================================================================
# 5. DIM USERS (SCD Type 2 via Auto CDC) -> silver schema
# =========================================================================

# Create the target streaming table in the silver schema
dlt.create_streaming_table(
    name="silver.dim_users",
    comment="Conformed Users SCD Type 2 dimension tracking user_type changes over time"
)

# Apply CDC changes into silver.dim_users
dlt.apply_changes(
    target="silver.dim_users",
    source="bronze_users",
    keys=["user_id"],
    sequence_by=F.coalesce(F.col("updated_at"), F.col("created_at"), F.col("_ingest_time")),
    stored_as_scd_type="2",
    track_history_column_list=["user_type", "country", "is_business", "company_name"]
)